# AutoGluon Local Pipeline: Relaxed Sentinel-2 + Context

Notebook นี้ออกแบบสำหรับรัน AutoGluon บนเครื่อง local โดยใช้ dataset หลัก `relaxed S2 + context`
และเก็บผลลัพธ์แบบ resumable ใต้ `E:\Water Quality Research\Experiments\autogluon_tabular\runs`

เป้าหมายรอบหลัก:

- Dataset: relaxed
- Feature set หลัก: `S2 + Context`
- Preset หลัก: `best_quality`
- Split หลัก: train 2562-2566, test 2567-2568
- Targets: regression ทั้งหมด ยกเว้น NH3 เป็น multiclass classification

หมายเหตุเชิงวิจัย:

- Turbidity และ SS เป็น optical-related targets ที่ Sentinel-2 มีโอกาสช่วยได้โดยตรงกว่า
- DO, BOD, TCB, FCB, NH3 เป็น indirect/proxy prediction ไม่ใช่ค่าที่ดาวเทียมวัดโดยตรง


## 0. Environment Check


In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import json
import math
import os
import shutil
import sys
import time
import traceback
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPECTED_ENV_FRAGMENT = r"E:\WQR\envs\rswq-ag-py311"
print("Python executable:", sys.executable)
if EXPECTED_ENV_FRAGMENT.lower() not in sys.executable.lower():
    raise RuntimeError(
        "Wrong Jupyter kernel. Please switch to kernel: Python (rswq-ag AutoGluon). "
        f"Current executable: {sys.executable}"
    )

from autogluon.tabular import TabularPredictor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

EXPECTED_ENV_FRAGMENT = r"E:\WQR\envs\rswq-ag-py311"
print("Python executable:", sys.executable)
if EXPECTED_ENV_FRAGMENT.lower() not in sys.executable.lower():
    print("WARNING: This notebook is not using the expected AutoGluon kernel.")
    print("Expected kernel: Python (rswq-ag AutoGluon)")

packages = [
    "autogluon.tabular",
    "autogluon.core",
    "pandas",
    "numpy",
    "scikit-learn",
    "xgboost",
    "lightgbm",
    "catboost",
    "torch",
]
for pkg in packages:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception as exc:
        print(f"{pkg}: VERSION_ERROR {exc}")


## 1. Config


In [ ]:
PROJECT_ROOT = Path(r"E:\Water Quality Research")
PROJECT_ALIAS = Path(r"E:\WQR")

DATA_PATHS = {
    "relaxed": PROJECT_ROOT / "Data" / "tabular_data_14" / "train" / "rswq_model_input_relaxed_s2_context_2562_2568.csv",
    "strict": PROJECT_ROOT / "Data" / "tabular_data_14" / "train" / "rswq_model_input_strict_s2_context_2562_2568.csv",
    "full": PROJECT_ROOT / "Data" / "tabular_data_14" / "train_full" / "rswq_model_input_full_s2_context_2562_2568.csv",
}

# Main recommended run.
DATASET_MODE = "relaxed"
RUN_MODE = "main_best_s2_context"
FEATURE_SETS_TO_RUN = ["S2 + Context"]
TARGETS_TO_RUN = ["Turbidity", "DO", "SS", "TCB", "BOD", "FCB", "NH3"]

# For smoke test, uncomment this block:
# RUN_MODE = "smoke_test"
# FEATURE_SETS_TO_RUN = ["S2 + Context"]
# TARGETS_TO_RUN = ["Turbidity"]
# PRESET = "medium_quality"
# TIME_LIMIT_PER_RUN = 300

PRESET = "best_quality"
TIME_LIMIT_PER_RUN = 1800  # 30 minutes per target-feature run.
RESUME = True
RANDOM_SEED = 42

# Use internal AutoGluon holdout/bagging on 2562-2566 by default.
# Alternative: "temporal_2566" uses 2562-2565 train, 2566 tuning, 2567-2568 test.
VALIDATION_MODE = "internal"

# FASTAI can be unstable on this local Windows setup; tree/boosting/ensemble models remain the main tabular baseline.
EXCLUDED_MODEL_TYPES = ["FASTAI"]

# Default keeps station/location memorization lower.
USE_LOCATION_FEATURES = False
USE_QA_AS_FEATURES = False

TRAIN_YEAR_MAX = 2566
TEMPORAL_VAL_YEAR = 2566
TEST_YEAR_MIN = 2567

RUN_TAG = f"ag_{DATASET_MODE}_{RUN_MODE}_{PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = PROJECT_ROOT / "Experiments" / "autogluon_tabular" / "runs" / RUN_TAG
DIRS = {
    "config": RUN_DIR / "config",
    "logs": RUN_DIR / "logs",
    "models": RUN_DIR / "models",
    "leaderboards": RUN_DIR / "leaderboards",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "feature_importance": RUN_DIR / "feature_importance",
    "plots": RUN_DIR / "plots",
    "summaries": RUN_DIR / "summaries",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_PATHS[DATASET_MODE]
print("DATA_PATH:", DATA_PATH)
print("RUN_DIR:", RUN_DIR)
print("PRESET:", PRESET)
print("TIME_LIMIT_PER_RUN:", TIME_LIMIT_PER_RUN)


## 2. Load Dataset


In [ ]:
assert DATA_PATH.exists(), f"Missing input dataset: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
display(df.head())

required_base_cols = ["sample_id", "year_be"]
missing_base = [c for c in required_base_cols if c not in df.columns]
assert not missing_base, f"Missing required columns: {missing_base}"

df["year_be"] = pd.to_numeric(df["year_be"], errors="coerce")
print("sample_id unique:", df["sample_id"].is_unique)
display(df["year_be"].value_counts(dropna=False).sort_index().rename("n").to_frame())


## 3. Target Definitions


In [ ]:
REGRESSION_TARGETS = {
    "DO": {"column": "DO_mg_l_clean", "label": "label_DO", "transform": "raw", "unit": "mg/L"},
    "BOD": {"column": "BOD_mg_l_clean", "label": "label_BOD", "transform": "raw", "unit": "mg/L"},
    "Turbidity": {"column": "turbidity_NTU_clean", "label": "label_Turbidity_log1p", "transform": "log1p", "unit": "NTU"},
    "SS": {"column": "SS_mg_l_clean", "label": "label_SS_log1p", "transform": "log1p", "unit": "mg/L"},
    "TCB": {"column": "TCB_MPN_100ml_clean", "label": "label_TCB_log1p", "transform": "log1p", "unit": "MPN/100mL"},
    "FCB": {"column": "FCB_MPN_100ml_clean", "label": "label_FCB_log1p", "transform": "log1p", "unit": "MPN/100mL"},
}

NH3_COL = "NH3_N_mg_l_clean"
NH3_LABEL_COL = "label_NH3_class"
NH3_LOW_THRESHOLD = 0.042426
NH3_HIGH_THRESHOLD = 0.120
NH3_CLASS_LABELS = ["low", "medium", "high"]

all_target_cols = [cfg["column"] for cfg in REGRESSION_TARGETS.values()] + [NH3_COL]
missing_targets = [c for c in all_target_cols if c not in df.columns]
assert not missing_targets, f"Missing target columns: {missing_targets}"

def target_transform(values, transform):
    values = np.asarray(values, dtype=float)
    if transform == "raw":
        return values
    if transform == "log1p":
        return np.log1p(values)
    raise ValueError(transform)

def inverse_target_transform(values, transform):
    values = np.asarray(values, dtype=float)
    if transform == "raw":
        return values
    if transform == "log1p":
        return np.clip(np.expm1(values), 0, None)
    raise ValueError(transform)

def make_nh3_class(series):
    values = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=series.index, dtype="object")
    out.loc[values <= NH3_LOW_THRESHOLD] = "low"
    out.loc[(values > NH3_LOW_THRESHOLD) & (values <= NH3_HIGH_THRESHOLD)] = "medium"
    out.loc[values > NH3_HIGH_THRESHOLD] = "high"
    return out

for target_name, cfg in REGRESSION_TARGETS.items():
    source = pd.to_numeric(df[cfg["column"]], errors="coerce")
    if cfg["transform"] == "log1p":
        source = source.where(source >= 0)
    df[cfg["label"]] = target_transform(source, cfg["transform"])

df[NH3_LABEL_COL] = make_nh3_class(df[NH3_COL])

target_summary = []
for target_name, cfg in REGRESSION_TARGETS.items():
    s = pd.to_numeric(df[cfg["column"]], errors="coerce")
    target_summary.append({
        "target": target_name,
        "task": "regression",
        "source_column": cfg["column"],
        "label_column": cfg["label"],
        "transform": cfg["transform"],
        "n": int(s.notna().sum()),
        "min": s.min(),
        "median": s.median(),
        "max": s.max(),
    })
s = pd.to_numeric(df[NH3_COL], errors="coerce")
target_summary.append({
    "target": "NH3",
    "task": "classification",
    "source_column": NH3_COL,
    "label_column": NH3_LABEL_COL,
    "transform": "low/medium/high",
    "n": int(s.notna().sum()),
    "min": s.min(),
    "median": s.median(),
    "max": s.max(),
})
display(pd.DataFrame(target_summary))
display(df[NH3_LABEL_COL].value_counts(dropna=False).rename("n").to_frame())


## 4. Feature Sets and Leakage Control


In [ ]:
S2_FEATURES = [
    "B1_median", "B2_median", "B3_median", "B4_median", "B5_median", "B6_median",
    "B7_median", "B8_median", "B8A_median", "B9_median", "B11_median", "B12_median",
    "NDWI_median", "MNDWI_median", "NDTI_median", "NDCI_median", "NDVI_median", "AWEIsh_median",
    "Red_Green_median", "Red_Blue_median", "NIR_Red_median",
    "B5_B4_median", "B6_B4_median", "B7_B4_median", "B8_B4_median",
    "ph_formula_s2_script_median", "turbidity_formula_s2_script_median", "salinity_formula_s2_script_median",
    "do_formula_s2_script_median", "chla_formula_s2_ndci_script_median", "secchi_formula_s2_script_median",
    "tsi_formula_s2_script_median",
]

CONTEXT_PREFIXES = ("rain_gsmap_", "rainy_hours_gsmap_", "era5h_", "jrc_", "srtm_", "wc_")
CONTEXT_EXACT = ["hours_since_last_rain_gsmap_mean", "month", "day_of_year", "doy_sin", "doy_cos"]
CONTEXT_CATEGORICAL = ["season_south_th"]
LOCATION_FEATURES = ["latitude", "longitude"]

QA_COLS = [
    "days_diff", "days_diff_signed", "abs_date_diff_days", "exact_date_match", "match_type",
    "scene_cloud_pct", "valid_pixel_count", "valid_water_pixel_count", "total_pixel_count",
    "water_pixel_fraction", "local_cloud_fraction", "local_cloud_pct", "water_pixel_pct",
    "candidate_valid_count", "candidate_water_count", "candidate_has_enough_water", "local_match_rank",
    "min_candidate_water_pixels", "has_image_match", "has_band_value", "has_clear_pixel",
    "has_water_pixel", "usable_s2", "usable_s2_water", "data_quality",
]

WQI_COLS = [c for c in df.columns if c == "WQI_clean" or c.startswith("WQI_")]
FIELD_AUX_COLS = ["pH_clean", "conductivity_clean", "salinity_ppt_clean", "temp_air_clean", "temp_water_clean"]
METADATA_COLS = [
    "sample_id", "station_survey_id", "canonical_station_survey_id", "station_original", "station_canonical",
    "survey_no", "field_date", "field_date_dt", "survey_time", "survey_time_clean",
    "sample_datetime_local", "sample_datetime_utc", "sample_cutoff_local", "sample_cutoff_utc",
    "sat_date", "sat_date_dt", "image_id", "source_chunk_file", "selection_reason",
    "water_body", "coordinate_waterbody", "coordinate_assignment_status", "station_location",
]
LABEL_COLS = [cfg["label"] for cfg in REGRESSION_TARGETS.values()] + [NH3_LABEL_COL]

LEAKAGE_DROP_COLS = sorted(
    set(all_target_cols)
    | set(LABEL_COLS)
    | set(WQI_COLS)
    | set(FIELD_AUX_COLS)
    | set(METADATA_COLS)
)

available_s2 = [c for c in S2_FEATURES if c in df.columns]
context_numeric = [c for c in df.columns if c.startswith(CONTEXT_PREFIXES) or c in CONTEXT_EXACT]
context_categorical = [c for c in CONTEXT_CATEGORICAL if c in df.columns]
context_location = [c for c in LOCATION_FEATURES if USE_LOCATION_FEATURES and c in df.columns]
available_context = context_numeric + context_categorical + context_location
available_qa = [c for c in QA_COLS if USE_QA_AS_FEATURES and c in df.columns]

FEATURE_SETS = {
    "S2 only": available_s2 + available_qa,
    "Context only": available_context + available_qa,
    "S2 + Context": available_s2 + available_context + available_qa,
}

for feature_set_name, cols in FEATURE_SETS.items():
    missing = [c for c in cols if c not in df.columns]
    leakage_overlap = sorted(set(cols) & set(LEAKAGE_DROP_COLS))
    duplicated = pd.Series(cols)[pd.Series(cols).duplicated()].tolist() if cols else []
    assert not missing, f"Missing feature columns in {feature_set_name}: {missing}"
    assert not leakage_overlap, f"Leakage columns in {feature_set_name}: {leakage_overlap}"
    assert not duplicated, f"Duplicate feature columns in {feature_set_name}: {duplicated}"

feature_summary = pd.DataFrame({
    "feature_set": list(FEATURE_SETS.keys()),
    "n_features": [len(cols) for cols in FEATURE_SETS.values()],
})
display(feature_summary)
print("S2 feature count:", len(available_s2))
print("Context feature count:", len(available_context))
print("QA feature count:", len(available_qa))


## 5. Temporal Split


In [ ]:
if VALIDATION_MODE == "internal":
    df["split_ag"] = np.where(
        df["year_be"] <= TRAIN_YEAR_MAX,
        "train",
        np.where(df["year_be"] >= TEST_YEAR_MIN, "test", "unknown"),
    )
elif VALIDATION_MODE == "temporal_2566":
    df["split_ag"] = np.select(
        [
            df["year_be"] < TEMPORAL_VAL_YEAR,
            df["year_be"] == TEMPORAL_VAL_YEAR,
            df["year_be"] >= TEST_YEAR_MIN,
        ],
        ["train", "val", "test"],
        default="unknown",
    )
else:
    raise ValueError(f"Unknown VALIDATION_MODE: {VALIDATION_MODE}")

display(df.groupby(["split_ag", "year_be"], dropna=False).size().reset_index(name="n"))


## 6. Helpers


In [ ]:
def safe_name(value):
    return str(value).replace(" ", "_").replace("+", "plus").replace("/", "_").replace("\\", "_")

def regression_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return {
            f"{prefix}n": 0,
            f"{prefix}mae": np.nan,
            f"{prefix}rmse": np.nan,
            f"{prefix}r2": np.nan,
            f"{prefix}pearson_r": np.nan,
            f"{prefix}spearman_r": np.nan,
            f"{prefix}bias": np.nan,
        }
    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}mae": mean_absolute_error(y_true, y_pred),
        f"{prefix}rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
        f"{prefix}r2": r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
        f"{prefix}pearson_r": pd.Series(y_true).corr(pd.Series(y_pred), method="pearson") if len(y_true) > 1 else np.nan,
        f"{prefix}spearman_r": pd.Series(y_true).corr(pd.Series(y_pred), method="spearman") if len(y_true) > 1 else np.nan,
        f"{prefix}bias": float(np.mean(y_pred - y_true)),
    }

def get_target_config(target_name):
    if target_name == "NH3":
        return {
            "task": "classification",
            "source_column": NH3_COL,
            "label_column": NH3_LABEL_COL,
            "transform": "class",
            "problem_type": "multiclass",
            "eval_metric": "f1_macro",
        }
    cfg = REGRESSION_TARGETS[target_name].copy()
    cfg.update({
        "task": "regression",
        "source_column": cfg["column"],
        "label_column": cfg["label"],
        "problem_type": "regression",
        "eval_metric": "root_mean_squared_error",
    })
    return cfg

def build_model_path(target_name, feature_set_name):
    return DIRS["models"] / f"{safe_name(target_name)}__{safe_name(feature_set_name)}"

def output_done_marker(target_name, feature_set_name):
    return DIRS["logs"] / f"DONE__{safe_name(target_name)}__{safe_name(feature_set_name)}.json"

def make_train_test_data(target_name, feature_set_name):
    cfg = get_target_config(target_name)
    label_col = cfg["label_column"]
    feature_cols = FEATURE_SETS[feature_set_name]
    keep_cols = feature_cols + [label_col, "sample_id", "year_be"]
    if "station_canonical" in df.columns:
        keep_cols.append("station_canonical")
    if "field_date" in df.columns:
        keep_cols.append("field_date")
    if "sat_date" in df.columns:
        keep_cols.append("sat_date")
    keep_cols = list(dict.fromkeys([c for c in keep_cols if c in df.columns]))

    valid_label = df[label_col].notna()
    train_mask = valid_label & (df["split_ag"] == "train")
    test_mask = valid_label & (df["split_ag"] == "test")

    train_data = df.loc[train_mask, keep_cols].copy()
    test_data = df.loc[test_mask, keep_cols].copy()

    tuning_data = None
    if VALIDATION_MODE == "temporal_2566":
        val_mask = valid_label & (df["split_ag"] == "val")
        tuning_data = df.loc[val_mask, keep_cols].copy()

    # AutoGluon should not see row identifiers or evaluation-only metadata as features.
    ag_drop_cols = ["sample_id", "year_be", "station_canonical", "field_date", "sat_date"]
    train_ag = train_data.drop(columns=[c for c in ag_drop_cols if c in train_data.columns])
    test_ag = test_data.drop(columns=[c for c in ag_drop_cols if c in test_data.columns])
    tuning_ag = None if tuning_data is None else tuning_data.drop(columns=[c for c in ag_drop_cols if c in tuning_data.columns])

    return train_data, test_data, tuning_data, train_ag, test_ag, tuning_ag

def save_run_config():
    config = {
        "run_tag": RUN_TAG,
        "run_dir": str(RUN_DIR),
        "data_path": str(DATA_PATH),
        "dataset_mode": DATASET_MODE,
        "run_mode": RUN_MODE,
        "targets_to_run": TARGETS_TO_RUN,
        "feature_sets_to_run": FEATURE_SETS_TO_RUN,
        "preset": PRESET,
        "time_limit_per_run": TIME_LIMIT_PER_RUN,
        "validation_mode": VALIDATION_MODE,
        "excluded_model_types": EXCLUDED_MODEL_TYPES,
        "use_location_features": USE_LOCATION_FEATURES,
        "use_qa_as_features": USE_QA_AS_FEATURES,
        "feature_sets": FEATURE_SETS,
        "leakage_drop_cols": LEAKAGE_DROP_COLS,
        "regression_targets": REGRESSION_TARGETS,
        "nh3_thresholds": {
            "low_threshold": NH3_LOW_THRESHOLD,
            "high_threshold": NH3_HIGH_THRESHOLD,
        },
    }
    path = DIRS["config"] / "run_config.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    return path

config_path = save_run_config()
print("Saved config:", config_path)


## 7. AutoGluon Training Loop


In [ ]:
all_regression_metrics = []
all_classification_metrics = []
all_regression_predictions = []
all_classification_predictions = []
run_log_rows = []

for target_name in TARGETS_TO_RUN:
    for feature_set_name in FEATURE_SETS_TO_RUN:
        if feature_set_name not in FEATURE_SETS:
            raise ValueError(f"Unknown feature set: {feature_set_name}")
        if target_name != "NH3" and target_name not in REGRESSION_TARGETS:
            raise ValueError(f"Unknown target: {target_name}")

        cfg = get_target_config(target_name)
        label_col = cfg["label_column"]
        run_name = f"{target_name}__{feature_set_name}"
        model_path = build_model_path(target_name, feature_set_name)
        done_marker = output_done_marker(target_name, feature_set_name)

        leaderboard_path = DIRS["leaderboards"] / f"leaderboard_{safe_name(run_name)}.csv"
        prediction_path = DIRS["predictions"] / f"predictions_{safe_name(run_name)}.csv"
        metric_path = DIRS["metrics"] / f"metrics_{safe_name(run_name)}.csv"
        fi_path = DIRS["feature_importance"] / f"feature_importance_{safe_name(run_name)}.csv"

        if RESUME and done_marker.exists() and leaderboard_path.exists() and prediction_path.exists() and metric_path.exists():
            print(f"SKIP completed run: {run_name}")
            run_log_rows.append({
                "target": target_name,
                "feature_set": feature_set_name,
                "status": "skipped_existing",
                "model_path": str(model_path),
            })
            continue

        print("=" * 100)
        print(f"Training AutoGluon: {run_name}")
        print("problem_type:", cfg["problem_type"], "eval_metric:", cfg["eval_metric"])
        print("model_path:", model_path)

        start_time = time.time()
        status = "success"
        error_message = None

        try:
            train_data, test_data, tuning_data, train_ag, test_ag, tuning_ag = make_train_test_data(target_name, feature_set_name)
            if train_ag.empty or test_ag.empty:
                raise ValueError(f"Empty train/test data: train={train_ag.shape}, test={test_ag.shape}")

            print("train_ag:", train_ag.shape, "test_ag:", test_ag.shape, "tuning_ag:", None if tuning_ag is None else tuning_ag.shape)

            fit_kwargs = {
                "train_data": train_ag,
                "presets": PRESET,
                "time_limit": TIME_LIMIT_PER_RUN,
                "verbosity": 2,
            }
            if tuning_ag is not None and not tuning_ag.empty:
                fit_kwargs["tuning_data"] = tuning_ag
                fit_kwargs["use_bag_holdout"] = True
            if EXCLUDED_MODEL_TYPES:
                fit_kwargs["excluded_model_types"] = EXCLUDED_MODEL_TYPES

            predictor = TabularPredictor(
                label=label_col,
                problem_type=cfg["problem_type"],
                eval_metric=cfg["eval_metric"],
                path=str(model_path),
            ).fit(**fit_kwargs)

            leaderboard = predictor.leaderboard(test_ag, silent=True)
            leaderboard.to_csv(leaderboard_path, index=False, encoding="utf-8-sig")

            y_pred = predictor.predict(test_ag.drop(columns=[label_col]))
            y_true_label = test_ag[label_col].copy()

            meta_cols = [c for c in ["sample_id", "year_be", "station_canonical", "field_date", "sat_date"] if c in test_data.columns]
            pred_df = test_data[meta_cols].copy()
            pred_df["target"] = target_name
            pred_df["feature_set"] = feature_set_name
            pred_df["best_model"] = predictor.model_best
            pred_df["label_column"] = label_col

            if cfg["task"] == "regression":
                source_col = cfg["source_column"]
                source_true = pd.to_numeric(df.loc[test_data.index, source_col], errors="coerce")
                y_pred_model = np.asarray(y_pred, dtype=float)
                y_true_model = np.asarray(y_true_label, dtype=float)
                y_pred_original = inverse_target_transform(y_pred_model, cfg["transform"])
                y_true_original = source_true.to_numpy(dtype=float)

                metric_row = {
                    "target": target_name,
                    "task": "regression",
                    "feature_set": feature_set_name,
                    "preset": PRESET,
                    "time_limit": TIME_LIMIT_PER_RUN,
                    "validation_mode": VALIDATION_MODE,
                    "best_model": predictor.model_best,
                    "n_features": len(FEATURE_SETS[feature_set_name]),
                    "n_train": int(len(train_ag)),
                    "n_test": int(len(test_ag)),
                    "transform": cfg["transform"],
                    "model_path": str(model_path),
                    "leaderboard_path": str(leaderboard_path),
                }
                metric_row.update(regression_metrics(y_true_original, y_pred_original, prefix="original_"))
                metric_row.update(regression_metrics(y_true_model, y_pred_model, prefix="modelscale_"))
                pd.DataFrame([metric_row]).to_csv(metric_path, index=False, encoding="utf-8-sig")
                all_regression_metrics.append(metric_row)

                pred_df["y_true_original"] = y_true_original
                pred_df["y_pred_original"] = y_pred_original
                pred_df["y_true_modelscale"] = y_true_model
                pred_df["y_pred_modelscale"] = y_pred_model
                pred_df["residual_original"] = pred_df["y_pred_original"] - pred_df["y_true_original"]
                all_regression_predictions.append(pred_df)

            else:
                y_true = y_true_label.astype(str)
                y_pred_label = pd.Series(y_pred, index=test_ag.index).astype(str)
                labels = NH3_CLASS_LABELS
                cm = confusion_matrix(y_true, y_pred_label, labels=labels)
                metric_row = {
                    "target": target_name,
                    "task": "classification",
                    "feature_set": feature_set_name,
                    "preset": PRESET,
                    "time_limit": TIME_LIMIT_PER_RUN,
                    "validation_mode": VALIDATION_MODE,
                    "best_model": predictor.model_best,
                    "n_features": len(FEATURE_SETS[feature_set_name]),
                    "n_train": int(len(train_ag)),
                    "n_test": int(len(test_ag)),
                    "accuracy": accuracy_score(y_true, y_pred_label),
                    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred_label),
                    "macro_f1": f1_score(y_true, y_pred_label, average="macro"),
                    "model_path": str(model_path),
                    "leaderboard_path": str(leaderboard_path),
                }
                pd.DataFrame([metric_row]).to_csv(metric_path, index=False, encoding="utf-8-sig")
                all_classification_metrics.append(metric_row)

                pred_df["y_true_label"] = y_true.values
                pred_df["y_pred_label"] = y_pred_label.values
                try:
                    proba = predictor.predict_proba(test_ag.drop(columns=[label_col]))
                    for col in proba.columns:
                        pred_df[f"prob_{col}"] = proba[col].values
                except Exception as exc:
                    print("predict_proba failed:", exc)

                cm_rows = []
                for i, true_label in enumerate(labels):
                    for j, pred_label in enumerate(labels):
                        cm_rows.append({
                            "target": target_name,
                            "feature_set": feature_set_name,
                            "true_label": true_label,
                            "pred_label": pred_label,
                            "n": int(cm[i, j]),
                        })
                pd.DataFrame(cm_rows).to_csv(DIRS["metrics"] / f"confusion_matrix_{safe_name(run_name)}.csv", index=False, encoding="utf-8-sig")
                all_classification_predictions.append(pred_df)

            pred_df.to_csv(prediction_path, index=False, encoding="utf-8-sig")

            try:
                fi = predictor.feature_importance(test_ag)
                fi.to_csv(fi_path, encoding="utf-8-sig")
            except Exception as exc:
                print("feature_importance failed:", exc)

            elapsed = time.time() - start_time
            with open(done_marker, "w", encoding="utf-8") as f:
                json.dump({
                    "target": target_name,
                    "feature_set": feature_set_name,
                    "status": "success",
                    "elapsed_seconds": elapsed,
                    "model_path": str(model_path),
                    "leaderboard_path": str(leaderboard_path),
                    "prediction_path": str(prediction_path),
                    "metric_path": str(metric_path),
                    "best_model": predictor.model_best,
                }, f, ensure_ascii=False, indent=2)

        except Exception as exc:
            status = "failed"
            error_message = repr(exc)
            traceback.print_exc()

        finally:
            elapsed = time.time() - start_time
            run_log_rows.append({
                "target": target_name,
                "feature_set": feature_set_name,
                "status": status,
                "error": error_message,
                "elapsed_seconds": elapsed,
                "model_path": str(model_path),
            })
            pd.DataFrame(run_log_rows).to_csv(DIRS["logs"] / "run_log.csv", index=False, encoding="utf-8-sig")

print("Training loop finished.")


## 8. Combine Outputs


In [ ]:
def read_existing_metrics(task):
    frames = []
    for p in DIRS["metrics"].glob("metrics_*.csv"):
        try:
            one = pd.read_csv(p)
        except Exception:
            continue
        if "task" in one.columns and len(one) and one["task"].iloc[0] == task:
            frames.append(one)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def read_existing_predictions(task):
    frames = []
    for p in DIRS["predictions"].glob("predictions_*.csv"):
        try:
            one = pd.read_csv(p)
        except Exception:
            continue
        if "target" not in one.columns:
            continue
        if task == "regression" and "y_true_original" in one.columns:
            frames.append(one)
        if task == "classification" and "y_true_label" in one.columns:
            frames.append(one)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

regression_metrics_df = read_existing_metrics("regression")
classification_metrics_df = read_existing_metrics("classification")
regression_predictions_df = read_existing_predictions("regression")
classification_predictions_df = read_existing_predictions("classification")

regression_metrics_df.to_csv(DIRS["metrics"] / "metrics_regression_all.csv", index=False, encoding="utf-8-sig")
classification_metrics_df.to_csv(DIRS["metrics"] / "metrics_classification_all.csv", index=False, encoding="utf-8-sig")
regression_predictions_df.to_csv(DIRS["predictions"] / "predictions_regression_all.csv", index=False, encoding="utf-8-sig")
classification_predictions_df.to_csv(DIRS["predictions"] / "predictions_classification_all.csv", index=False, encoding="utf-8-sig")

if not regression_metrics_df.empty:
    display(regression_metrics_df.sort_values(["target", "original_rmse", "original_mae"]))
if not classification_metrics_df.empty:
    display(classification_metrics_df.sort_values(["macro_f1", "balanced_accuracy"], ascending=False))


## 9. Plots


In [ ]:
def plot_best_regression(metrics_df, predictions_df):
    if metrics_df.empty or predictions_df.empty:
        print("No regression results to plot.")
        return
    best_rows = (
        metrics_df
        .sort_values(["target", "original_rmse", "original_mae"])
        .groupby("target", as_index=False)
        .head(1)
    )
    n = len(best_rows)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for ax, (_, row) in zip(axes, best_rows.iterrows()):
        mask = (
            (predictions_df["target"] == row["target"])
            & (predictions_df["feature_set"] == row["feature_set"])
            & (predictions_df["best_model"] == row["best_model"])
        )
        plot_df = predictions_df.loc[mask].copy()
        if plot_df.empty:
            continue
        ax.scatter(plot_df["y_true_original"], plot_df["y_pred_original"], alpha=0.75, s=28)
        lo = np.nanmin([plot_df["y_true_original"].min(), plot_df["y_pred_original"].min()])
        hi = np.nanmax([plot_df["y_true_original"].max(), plot_df["y_pred_original"].max()])
        ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1)
        ax.set_title(
            f"{row['target']} | {row['best_model']} | {row['feature_set']}\n"
            f"RMSE={row['original_rmse']:.3g}, R2={row['original_r2']:.3g}, r={row['original_pearson_r']:.3g}"
        )
        ax.set_xlabel("Observed")
        ax.set_ylabel("Predicted")
        ax.grid(True, alpha=0.3)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    out = DIRS["plots"] / "observed_vs_predicted_best_regression.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(out)
    plt.show()

def plot_nh3_confusion(classification_metrics_df, classification_predictions_df):
    if classification_metrics_df.empty or classification_predictions_df.empty:
        print("No classification results to plot.")
        return
    best = classification_metrics_df.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).iloc[0]
    sub = classification_predictions_df[
        (classification_predictions_df["target"] == "NH3")
        & (classification_predictions_df["feature_set"] == best["feature_set"])
        & (classification_predictions_df["best_model"] == best["best_model"])
    ]
    if sub.empty:
        print("No NH3 predictions for best classification model.")
        return
    cm = confusion_matrix(sub["y_true_label"], sub["y_pred_label"], labels=NH3_CLASS_LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=NH3_CLASS_LABELS)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(
        f"NH3 | {best['best_model']} | {best['feature_set']}\n"
        f"Macro F1={best['macro_f1']:.3g}, Balanced acc={best['balanced_accuracy']:.3g}"
    )
    fig.tight_layout()
    out = DIRS["plots"] / "nh3_confusion_matrix_best.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(out)
    plt.show()

plot_best_regression(regression_metrics_df, regression_predictions_df)
plot_nh3_confusion(classification_metrics_df, classification_predictions_df)


## 10. Summary Export


In [ ]:
summary_lines = [
    f"# AutoGluon Run Summary",
    "",
    f"- Run tag: `{RUN_TAG}`",
    f"- Dataset: `{DATASET_MODE}`",
    f"- Data path: `{DATA_PATH}`",
    f"- Preset: `{PRESET}`",
    f"- Time limit per run: `{TIME_LIMIT_PER_RUN}` seconds",
    f"- Validation mode: `{VALIDATION_MODE}`",
    f"- Feature sets: `{FEATURE_SETS_TO_RUN}`",
    f"- Targets: `{TARGETS_TO_RUN}`",
    "",
]

if not regression_metrics_df.empty:
    summary_lines.append("## Best Regression Models")
    best_reg = (
        regression_metrics_df
        .sort_values(["target", "original_rmse", "original_mae"])
        .groupby("target", as_index=False)
        .head(1)
    )
    summary_lines.append(best_reg[[
        "target", "feature_set", "best_model", "original_mae", "original_rmse",
        "original_r2", "original_pearson_r", "original_spearman_r", "original_bias",
    ]].to_markdown(index=False))
    summary_lines.append("")

if not classification_metrics_df.empty:
    summary_lines.append("## Best Classification Models")
    best_cls = classification_metrics_df.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).head(5)
    summary_lines.append(best_cls[[
        "target", "feature_set", "best_model", "accuracy", "balanced_accuracy", "macro_f1",
    ]].to_markdown(index=False))
    summary_lines.append("")

summary_lines.extend([
    "## Interpretation Notes",
    "",
    "- Turbidity and SS are optical-related targets and are the strongest candidates for future image patch/CNN work.",
    "- DO, BOD, TCB, FCB, and NH3 should be interpreted as indirect/proxy or risk-screening models.",
    "- Keep temporal test years 2567-2568 as unseen test evidence.",
])

summary_path = DIRS["summaries"] / "run_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(summary_path)


## 11. Recommended Run Order

1. Smoke test:

```python
RUN_MODE = "smoke_test"
TARGETS_TO_RUN = ["Turbidity"]
FEATURE_SETS_TO_RUN = ["S2 + Context"]
PRESET = "medium_quality"
TIME_LIMIT_PER_RUN = 300
```

2. Main run:

```python
RUN_MODE = "main_best_s2_context"
TARGETS_TO_RUN = ["Turbidity", "DO", "SS", "TCB", "BOD", "FCB", "NH3"]
FEATURE_SETS_TO_RUN = ["S2 + Context"]
PRESET = "best_quality"
TIME_LIMIT_PER_RUN = 1800
```

3. Optional diagnostic:

```python
FEATURE_SETS_TO_RUN = ["S2 only", "Context only"]
PRESET = "high_quality"
TIME_LIMIT_PER_RUN = 600
```
